# 03 - Modelleme

Bu defter: veri bölme, baseline, model karşılaştırma, hiperparametre ayarı, özellik önemi ve hata analizi. Tüm fiyatlar USD.

**Hazırlık.** Defter `notebooks/` içinden çalışsa da proje köküne geçilir ki `from src...` ve `data/cars.db` yolları doğru çalışsın. Grafikler ekrana değil dosyaya çizileceği için matplotlib `Agg` arka ucu seçilir.

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
import matplotlib
matplotlib.use('Agg')
print(os.getcwd())

C:\Users\KAAN\Desktop\car-price-prediction\repository


**Veriyi yükle ve böl.** Veritabanındaki temiz `cars` tablosu okunur. %20'si hiç eğitimde kullanılmayacak 'sınav' seti olarak ayrılır (`random_state=42` ile her seferinde aynı).

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from src.db.queries import load_cars
from src.ml.config import FEATURES, RANDOM_STATE, TARGET

df = load_cars()
X, y = df[FEATURES], df[TARGET]
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print(len(df), len(X_train), len(X_val))

771 616 155


**Dört model karşılaştırması.** Baseline (medyan), Ridge, RandomForest ve HistGB; her biri eğitim setinde fit edilip holdout'ta ölçülür, ayrıca tüm veride 5 katlı CV yapılır. Veri az (771 satır) olduğundan model seçimi CV MAE'ye göre yapılır; holdout ek bilgidir.

In [3]:
from src.ml.evaluate import cv_scores, evaluate
from src.ml.models import (make_baseline, make_hist_gb, make_linear_model,
                           make_random_forest)

candidates = {
    'baseline': make_baseline(),
    'ridge': make_linear_model(),
    'random_forest': make_random_forest(),
    'hist_gb': make_hist_gb(),
}
rows = []
for name, model in candidates.items():
    hold = evaluate(model.fit(X_train, y_train), X_val, y_val)
    cv = cv_scores(model, X, y)
    rows.append({'model': name, 'hold_mae': hold['mae'], 'hold_rmse': hold['rmse'],
                 'hold_r2': hold['r2'], 'cv_mae': cv['mae_mean'], 'cv_mae_std': cv['mae_std'],
                 'cv_rmse': cv['rmse_mean'], 'cv_r2': cv['r2_mean']})
results = pd.DataFrame(rows)
print(results.round(3).to_string(index=False))

        model  hold_mae  hold_rmse  hold_r2    cv_mae  cv_mae_std   cv_rmse  cv_r2
     baseline  9094.097  15965.655   -0.123 10083.561    1002.318 17857.114 -0.120
        ridge  7441.622  13948.469    0.143  7304.534     550.840 14056.341  0.304
random_forest  5854.303  12399.646    0.323  6048.071     588.576 12516.047  0.448
      hist_gb  5965.828  12576.126    0.303  6027.805     791.714 12182.492  0.475


**En iyi ağaç modelini seç ve ayarla.** CV MAE'si düşük olan ağaç modeli (RF veya HistGB) için `RandomizedSearchCV` ile 15 rastgele ayar kombinasyonu denenir. Parametre adları `regressor__reg__...` biçimindedir çünkü model `TransformedTargetRegressor > Pipeline > reg` yapısındadır.

In [4]:
import numpy as np
from sklearn.model_selection import KFold, RandomizedSearchCV

tree = results[results.model.isin(['random_forest', 'hist_gb'])].sort_values('cv_mae')
best_name = tree.iloc[0]['model']
print('Ayarlanacak model:', best_name)
grids = {
    'hist_gb': {
        'regressor__reg__learning_rate': [0.03, 0.05, 0.1],
        'regressor__reg__max_iter': [200, 400, 800],
        'regressor__reg__max_leaf_nodes': [15, 31, 63],
        'regressor__reg__min_samples_leaf': [5, 10, 20],
        'regressor__reg__l2_regularization': [0, 0.1, 1],
    },
    'random_forest': {
        'regressor__reg__max_depth': [None, 10, 20],
        'regressor__reg__min_samples_leaf': [1, 2, 4],
        'regressor__reg__max_features': [1.0, 0.5, 'sqrt'],
    },
}
makers = {'hist_gb': make_hist_gb, 'random_forest': make_random_forest}
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
search = RandomizedSearchCV(makers[best_name](), grids[best_name], n_iter=15, cv=cv,
                            scoring='neg_mean_absolute_error', random_state=RANDOM_STATE,
                            n_jobs=-1)
search.fit(X, y)
print('Arama en iyi CV MAE:', round(-search.best_score_, 1))
print(search.best_params_)

Ayarlanacak model: hist_gb


Arama en iyi CV MAE: 5881.2
{'regressor__reg__min_samples_leaf': 5, 'regressor__reg__max_leaf_nodes': 15, 'regressor__reg__max_iter': 400, 'regressor__reg__learning_rate': 0.03, 'regressor__reg__l2_regularization': 0}


**Ayarlı modeli tabloya ekle ve karar ver.** Ayarlı model aynı CV bölmeleriyle varsayılandan daha iyiyse onu, değilse varsayılanı seçeriz. Not: arama ve CV aynı veride yapıldığı için ayarlı skor hafif iyimser olabilir.

In [5]:
params = {k.replace('regressor__reg__', ''): v for k, v in search.best_params_.items()}
tuned = makers[best_name](**params)
hold = evaluate(tuned.fit(X_train, y_train), X_val, y_val)
cvs = cv_scores(tuned, X, y)
results.loc[len(results)] = [best_name + '_ayarli', hold['mae'], hold['rmse'], hold['r2'],
                             cvs['mae_mean'], cvs['mae_std'], cvs['rmse_mean'], cvs['r2_mean']]
print(results.round(3).to_string(index=False))
default_cv = float(results.loc[results.model == best_name, 'cv_mae'].iloc[0])
use_tuned = cvs['mae_mean'] < default_cv
chosen_params = params if use_tuned else {}
print('Ayarlı seçildi mi:', use_tuned, '| model:', best_name, '| parametreler:', chosen_params)

         model  hold_mae  hold_rmse  hold_r2    cv_mae  cv_mae_std   cv_rmse  cv_r2
      baseline  9094.097  15965.655   -0.123 10083.561    1002.318 17857.114 -0.120
         ridge  7441.622  13948.469    0.143  7304.534     550.840 14056.341  0.304
 random_forest  5854.303  12399.646    0.323  6048.071     588.576 12516.047  0.448
       hist_gb  5965.828  12576.126    0.303  6027.805     791.714 12182.492  0.475
hist_gb_ayarli  5445.348  11854.543    0.381  5881.221     807.690 12122.876  0.481
Ayarlı seçildi mi: True | model: hist_gb | parametreler: {'min_samples_leaf': 5, 'max_leaf_nodes': 15, 'max_iter': 400, 'learning_rate': 0.03, 'l2_regularization': 0}


**Özellik önemi (permutation importance).** Seçilen model train'de eğitilir; holdout'ta her sütun sırayla karıştırılır ve MAE'nin ne kadar arttığına bakılır. Çok artıyorsa model o sütuna çok güveniyordur. Sonuç `docs/img/ozellik_onemi.png` dosyasına kaydedilir.

In [6]:
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance

final = makers[best_name](**chosen_params).fit(X_train, y_train)
perm = permutation_importance(final, X_val, y_val, n_repeats=5, random_state=RANDOM_STATE,
                              scoring='neg_mean_absolute_error', n_jobs=-1)
importance = pd.Series(perm.importances_mean, index=X_val.columns).sort_values()
fig, ax = plt.subplots(figsize=(7, 4.5))
importance.plot.barh(ax=ax, xerr=pd.Series(perm.importances_std, index=X_val.columns)[importance.index])
ax.set_title('Özellik önemi (karıştırınca MAE artışı, USD)')
ax.set_xlabel('MAE artışı (USD)')
fig.savefig('docs/img/ozellik_onemi.png', dpi=150, bbox_inches='tight')
print(importance.sort_values(ascending=False).round(1))

year            3337.2
mileage         1410.6
cylinders        654.3
brand            451.8
drive            451.2
fuel_type        423.0
condition        303.5
body_type        155.0
title_status      36.0
transmission     -47.6
model            -99.0
dtype: float64


**Hata analizi.** Holdout tahminleri gerçek fiyatlarla yan yana konur. (1) En az 5 ilanı olan markalarda ortalama mutlak hata; (2) fiyat beş eşit dilime bölünüp her dilimde medyan oransal hata (|hata| / gerçek fiyat). Böylece modelin nerede zayıf olduğu görülür.

In [7]:
err = X_val.assign(actual=y_val, pred=final.predict(X_val))
err['abs_err'] = (err['actual'] - err['pred']).abs()
by_brand = (err.groupby('brand')['abs_err'].agg(['mean', 'count']).query('count >= 5')
            .sort_values('mean', ascending=False).head(10))
print(by_brand.round(0))
err['price_bin'] = pd.qcut(err['actual'], 5)
err['rel_err'] = err['abs_err'] / err['actual']
by_bin = err.groupby('price_bin', observed=True)['rel_err'].median()
print(by_bin.round(3))

                 mean  count
brand                       
Mercedes-Benz  7604.0      7
Chevrolet      6721.0     14
Ford           5602.0     24
Toyota         4733.0     23
Honda          3829.0      9
BMW            3048.0     11
Nissan         2780.0      6
Subaru         2376.0      5
Hyundai        1828.0      9
price_bin
(1199.999, 4999.8]     0.316
(4999.8, 7959.4]       0.203
(7959.4, 12500.0]      0.267
(12500.0, 23180.0]     0.151
(23180.0, 120000.0]    0.312
Name: rel_err, dtype: float64


Not: holdout 155 satır olduğundan bazı marka sayıları küçük kalabilir; marka tablosu yalnızca yön göstericidir. Aynı analiz tüm veride CV tahminleriyle de yapılabilir.